# An Inefficient Vector Space Model

In [1]:
# Import defaultdict from the collections module.
# defaultdict is a dictionary-like object which provides a default value for a key that does not exist.
from collections import defaultdict

# Import log and sqrt functions from the math module.
# log is used for calculating logarithms and sqrt for calculating square roots.
from math import log, sqrt

# Import the re module for regular expression operations.
# This module provides support for regular expressions (pattern matching in strings).
import re


The dataset is the TIME dataset, available at http://ir.dcs.gla.ac.uk/resources/test_collections/time/

In [2]:
def import_dataset():
    """
    This function imports all the articles in the TIME corpus,
    returning a list of lists where each sub-list contains all the
    terms present in the document as a string.
    """
    # Initialize an empty list to store the articles.
    articles = []

    # Open the 'TIME.ALL' file in read mode.
    with open('TIME.ALL', 'r') as f:
        # Initialize a temporary list to store words of the current article.
        tmp = []

        # Iterate over each row in the file.
        for row in f:
            # Check if the row starts with '*TEXT', indicating a new article.
            if row.startswith("*TEXT"):
                # If tmp is not empty, it means we have reached a new article.
                # Add the previous article's words to the articles list.
                if tmp != []:
                    articles.append(tmp)
                # Reset tmp for the new article.
                tmp = []
            else:
                # Remove any non-alphabetic characters and split the row into words.
                # Regular expression is used here to replace non-letter characters with nothing ('').
                row = re.sub(r'[^a-zA-Z\s]+', '', row)
                # Extend the temporary list with the words from this row.
                tmp += row.split()

        # If the last article was not followed by a new '*TEXT', add it to the articles list.
        if tmp != []:
            articles.append(tmp)

    # Return the list of articles.
    return articles

In [3]:
def make_inverted_index(articles):
    """
    This function builds an inverted index as a hash table (dictionary)
    where the keys are the terms and the values are ordered sets of
    docIDs containing the term.
    """
    # Create a defaultdict where each key will have a set as its default value.
    # This is used to store the inverted index.
    index = defaultdict(set)

    # Enumerate over the articles, getting both the index (docid) and the article.
    # The index acts as a unique identifier for each document.
    for docid, article in enumerate(articles):
        # Iterate over each term in the article.
        for term in article:
            # Add the document ID to the set of docIDs for this term.
            # Since it's a set, each docID will be unique per term.
            index[term].add(docid)

    # Return the constructed inverted index.
    return index

In [4]:
index = defaultdict(set)

In [5]:
index["home"].add(1)
index["home"].add(2)
index["home"].add(2)
index["yes"].add(2)
index["more"].add(3)

In [6]:
index

defaultdict(set, {'home': {1, 2}, 'yes': {2}, 'more': {3}})

In [7]:
def make_positional_index(articles):
    """
    A more advanced version of make_inverted_index.
    This function builds a positional inverted index as a dictionary. 
    Here, for each term in the articles, the index stores a dictionary 
    where the keys are document IDs and the values are lists of positions 
    where the term appears in the document.
    """
    # Create a defaultdict of dictionaries. The outer dictionary holds terms,
    # and the inner dictionary maps document IDs to a list of positions.
    index = defaultdict(dict)

    # Enumerate over the articles to get both the document ID (docid) and the article itself.
    for docid, article in enumerate(articles):
        # Enumerate over each term in the article to get both the position (pos) and the term.
        for pos, term in enumerate(article):
            try:
                # Try to append the position to the existing list for this term in this document.
                index[term][docid].append(pos)
            except KeyError:
                # If the term or document ID doesn't exist yet in the index,
                # create a new entry with the current position in a list.
                index[term][docid] = [pos]

    # Return the constructed positional inverted index.
    return index

In [8]:
articles = [["home","no","yes"],[ "more", "advanced", "version","no","yes","yes" ]]

In [67]:
p_index = make_positional_index(articles)

In [68]:
p_index.keys()

dict_keys(['THE', 'ALLIES', 'AFTER', 'NASSAU', 'IN', 'DECEMBER', 'US', 'FIRST', 'PROPOSED', 'TO', 'HELP', 'NATO', 'DEVELOP', 'ITS', 'OWN', 'NUCLEAR', 'STRIKE', 'FORCE', 'BUT', 'EUROPE', 'MADE', 'NO', 'ATTEMPT', 'DEVISE', 'A', 'PLAN', 'LAST', 'WEEK', 'AS', 'THEY', 'STUDIED', 'ACCORD', 'BETWEEN', 'PRESIDENT', 'KENNEDY', 'AND', 'PRIME', 'MINISTER', 'MACMILLAN', 'EUROPEANS', 'SAW', 'EMERGING', 'OUTLINES', 'OF', 'THAT', 'WANTS', 'WILL', 'SUPPORT', 'IT', 'ALL', 'SPRANG', 'FROM', 'ANGLOUS', 'CRISIS', 'OVER', 'CANCELLATION', 'BUGRIDDEN', 'SKYBOLT', 'MISSILE', 'OFFER', 'SUPPLY', 'BRITAIN', 'FRANCE', 'WITH', 'PROVED', 'POLARIS', 'TIME', 'DEC', 'ONE', 'ALLIED', 'LEADER', 'WHO', 'UNRESERVEDLY', 'WELCOMED', 'WAS', 'HAROLD', 'BY', 'THUS', 'KEEPING', 'SEPARATE', 'DETERRENT', 'FOR', 'HAD', 'SAVED', 'HIS', 'NECK', 'BACK', 'BEAMED', 'NOW', 'WEAPON', 'GENERATION', 'TERMS', 'ARE', 'VERY', 'GOOD', 'MANY', 'OTHER', 'BRITONS', 'WERE', 'NOT', 'SO', 'SURE', 'THOUGH', 'GOVERNMENT', 'SHOULDER', 'NONE', 'MILLION'

In [43]:
def documents_as_vectors(articles, normalized=True):
    """
    This function generates a list of dictionaries, where each dictionary represents 
    the TF-IDF vector of a document. Each term's TF-IDF value is calculated and 
    stored in the dictionary. This function is suitable for small collections 
    as its space complexity is O(#documents x #terms).
    """
    # Generate a positional index from the articles
    p_index = make_positional_index(articles)

    # Initialize an empty list to store the TF-IDF vectors of each document
    vectors = []

    # Calculate the total number of documents
    n = len(articles)

    # Calculate the Inverse Document Frequency (IDF) for each term
    idf = {}
    for term in p_index.keys():
        idf[term] = log(n / len(p_index[term]))

    # Iterate over each document to create its TF-IDF vector
    for docid in range(0, len(articles)):
        # Initialize an empty dictionary for the TF-IDF vector of the current document
        v = {}

        # Calculate TF-IDF for each term and store it in the vector
        for term in p_index.keys():
            try:
                # Term Frequency (TF) is the number of times a term occurs in this document
                # Multiply TF with IDF to get TF-IDF
                tfidf = len(p_index[term][docid]) * idf[term]
            except KeyError:
                # If the term is not in the document, its TF-IDF is 0
                tfidf = 0

            # Assign the TF-IDF score to the term in the vector
            v[term] = tfidf
        
        if normalized:
            norm = sqrt(sum(x**2 for x in v.values()))
            if norm != 0:
                for term in v:
                    v[term] /= norm

        # Add the document's TF-IDF vector to the list
        vectors.append(v)

    # Return the list of TF-IDF vectors
    return vectors

In [50]:
# Example of usage
articles = import_dataset()
vectors = documents_as_vectors(articles, True)

In [51]:
vectors[1]

{'THE': 0.0,
 'ALLIES': 0.0,
 'AFTER': 0.014925594672354435,
 'NASSAU': 0.0,
 'IN': 0.0002759047586412043,
 'DECEMBER': 0.0,
 'US': 0.0,
 'FIRST': 0.0,
 'PROPOSED': 0.0,
 'TO': 0.0002759047586412043,
 'HELP': 0.0,
 'NATO': 0.0,
 'DEVELOP': 0.0,
 'ITS': 0.0,
 'OWN': 0.0,
 'NUCLEAR': 0.0,
 'STRIKE': 0.0,
 'FORCE': 0.0,
 'BUT': 0.0,
 'EUROPE': 0.0,
 'MADE': 0.0,
 'NO': 0.014103872920778913,
 'ATTEMPT': 0.0,
 'DEVISE': 0.0,
 'A': 0.0,
 'PLAN': 0.05122596464614421,
 'LAST': 0.004510297651237541,
 'WEEK': 0.0072612653486015515,
 'AS': 0.003889679130758158,
 'THEY': 0.0,
 'STUDIED': 0.0,
 'ACCORD': 0.0,
 'BETWEEN': 0.0,
 'PRESIDENT': 0.0,
 'KENNEDY': 0.0,
 'AND': 0.0004978086756246559,
 'PRIME': 0.0,
 'MINISTER': 0.0,
 'MACMILLAN': 0.0,
 'EUROPEANS': 0.0,
 'SAW': 0.0,
 'EMERGING': 0.0,
 'OUTLINES': 0.0,
 'OF': 0.0,
 'THAT': 0.002723570151218975,
 'WANTS': 0.0,
 'WILL': 0.0,
 'SUPPORT': 0.0,
 'IT': 0.01507174272211624,
 'ALL': 0.010294739653769355,
 'SPRANG': 0.0,
 'FROM': 0.003954892994877695

In [ ]:
def show_document_vector(v, docid):
    """
    This function prints the terms and their corresponding non-zero TF-IDF weights 
    (both normalized and unnormalized) for a given document represented as a vector in v.
    """
    # Create a list of terms with non-zero weights in the specified document's vector.
    non_zero_terms = [x for x in v[docid].keys() if v[docid][x] > 0]

    # Create a list of tuples (term, TF-IDF weight) for the non-zero terms.
    vector = [(x, v[docid][x]) for x in non_zero_terms]

    # Sort the vector in descending order based on the TF-IDF weights.
    vector.sort(key=lambda x: x[1], reverse=True)

    # Calculate the length of the vector (Euclidean norm) for normalization.
    length = sqrt(sum([x[1]**2 for x in vector]))

    # Normalize the vector by dividing each TF-IDF weight by the vector's length.
    normalized = {k: tfidf/length for k, tfidf in vector}

    # Print each term along with its unnormalized and normalized TF-IDF weight.
    for (term, tfidf) in vector:
        print(f"{term}:\t{tfidf}\t(normalized: {normalized[term]})")


In [16]:
" ".join(articles[2])

'BERLIN ONE LAST RUN HANS WEIDNER HAD BEEN HOPING FOR MONTHS TO ESCAPE DRAB EAST GERMANY AND MAKE HIS WAY TO THE WEST THE ODDS WERE AGAINST HIM FOR WEIDNER WAS A CRIPPLE ON CRUTCHES WHO LIVED IN THE VILLAGE OF NEUGERSDORF MILES SOUTHEAST OF THE FRONTIER OF FREEDOM BUT HANS WEIDNER DID HAVE ONE MAJOR ASSET THE BUS THAT HE OPERATED FOR THE LOCAL COMMUNIST REGIME IT WAS AN UGLY THING AND ANCIENT ITS CHASSIS CREAKED AND THE ENGINE COUGHED A CREAMCOLORED COAT OF PAINT COULD NOT DISGUISE THE WELTS AND BRUISES OF TWO DECADES OF CHUGGING SERVICE IN FACT THE BUS WAS READY FOR THE JUNK PILE WHEN WEIDNER DECIDED TO PRESS IT INTO SERVICE FOR ONE LAST RUN SHARP BLADES THE HAZARDS WOULD BE GREAT ON THE JOURNEY TO THE BORDER SO WEIDNER SIGNED UP A FELLOW VILLAGER JURGEN WAGNER TO TAKE THE WHEEL EIGHT DAYS BEFORE CHRISTMAS THE PAIR BEGAN THE FEVERISH PREPARATIONS IN WEIDNERS GARAGE FIRST WEIDNER AND WAGNER ATTACHED A HEAVY SNOWPLOW TO THE FRONT OF THE BUS NOT TO PLOW SNOW BUT TO SCOOP AWAY THE HEAVY O

In [49]:
show_document_vector(vectors, 2)

WEIDNER:	48.37897743237022	(normalized: 0.5344367558270138)
BUS:	25.54633834373535	(normalized: 0.28220733286003796)
WAGNER:	24.18948871618511	(normalized: 0.2672183779135069)
POTATOES:	13.313802799836534	(normalized: 0.14707598121543528)
WAHAH:	12.094744358092555	(normalized: 0.13360918895675344)
BERLIN:	11.825318902751848	(normalized: 0.1306328782959754)
BLADES:	10.708449996972666	(normalized: 0.11829496157330173)
AUTOBAHN:	9.897519780756337	(normalized: 0.1093367128264654)
HANS:	8.875868533224356	(normalized: 0.09805065414362352)
WHEEL:	8.875868533224356	(normalized: 0.09805065414362352)
CHECKPOINT:	8.875868533224356	(normalized: 0.09805065414362352)
CHRISTMAS:	7.9358612747328845	(normalized: 0.08766650680639829)
STOP:	7.744908828739654	(normalized: 0.08555707805924224)
YARDS:	7.700295203420117	(normalized: 0.08506423669617734)
STEEL:	6.816629698862039	(normalized: 0.07530248995085918)
SHARP:	6.678643955888136	(normalized: 0.07377817801333546)
WIVES:	6.549566913612994	(normalized: 0

In [20]:
len(vectors)

423